# Beat Timer v0.5 Development Notebook

Use this notebook to change and inspect the audio-only predictor. Map data is deliberately absent here so accidental leakage has fewer opportunities to crawl out of the floorboards.

Keep this notebook in the same directory as `beattimer_core.py`.

After editing the Python module, rerun the reload cell before analyzing the song again.

In [ ]:
from pathlib import Path
import numpy as np
import importlib

import beattimer_core as core

In [ ]:
import inspect
from pathlib import Path

print("Loaded module from:")
print(Path(core.__file__).resolve())

print()
print("Minimum windows setting:")
print(core.LOCAL_MINIMUM_WINDOWS_PER_SECTION)

print()
print("Segmenter signature:")
print(inspect.signature(core.segment_local_tempo_windows))

print()
print("Has trusted-window code:")
source = inspect.getsource(
    core.segment_local_tempo_windows
)

print(
    "valid_for_segmentation" in source,
    "global fallback" in source,
    "reference_bpm" in source,
)

In [ ]:
# Reload after editing beattimer_core.py.
importlib.reload(core)

## Inputs and development settings

In [ ]:
song_selection = "reign_of_fear"

AUDIO_PATH = Path(Path("songs") / song_selection / "song.mp3")
OSU_PATH = Path(Path("songs") / song_selection / "song.osu")

ANALYSIS_SETTINGS = {
    "round_initial_bpm": 1,
    "use_local_tempo": True,
    "integer_section_bpm": True,
    "section_tolerance_ms": 6.0,
}

## Run the audio-only predictor

In [ ]:
result = core.analyze_song(
    AUDIO_PATH,
    **ANALYSIS_SETTINGS,
)

core.print_analysis_summary(result)

In [ ]:
print("Timing sections:", len(result["sections"]))

for index, section in enumerate(
    result["sections"],
    start=1,
):
    print(
        f"{index:2d} | "
        f"{section['start_time']:7.3f}"
        f"–{section['end_time']:7.3f}s | "
        f"{section['bpm']:6.1f} BPM | "
        f"source={section.get('source', 'MISSING')} | "
        f"windows={section['window_count']}"
    )

In [ ]:
accepted_windows = [
    window
    for window in result["local_windows"]
    if window.get("accepted", False)
]

fallback_windows = [
    window
    for window in result["local_windows"]
    if window.get("used_fallback", False)
]

print("Accepted local windows:", len(accepted_windows))
print("Fallback local windows:", len(fallback_windows))

reasons = {}

for window in fallback_windows:
    reason = window.get(
        "rejection_reason",
        "unknown",
    )
    reasons[reason] = reasons.get(reason, 0) + 1

print("\nFallback reasons:")

for reason, count in sorted(
    reasons.items(),
    key=lambda item: item[1],
    reverse=True,
):
    print(f"{count:3d} | {reason}")

In [ ]:
trusted = [
    window
    for window in result["local_windows"]
    if window.get(
        "valid_for_segmentation",
        False,
    )
]

print("Trusted windows:", len(trusted))

for window in trusted:
    print(
        f"{window['center_time']:6.2f}s | "
        f"{window['candidate_bpm']:6.1f} BPM | "
        f"confidence={window['confidence']:.3f}"
    )

## Interactive timing view

In [ ]:
# figure = core.visualize_timing_segments(result)

## Focused passage

In [ ]:
# focused_figure = core.visualize_timing_segments(
#     result,
#     time_range=(45, 65),
#     pixels_per_second=70,
# )

## Optional global-mode tolerance sweep

In [ ]:
# sweep_results = core.tolerance_sweep(result)

## Development notes

The next target is the local tracker in:

- `estimate_local_tempo_windows`
- `segment_local_tempo_windows`

Do not import `.osu` information into this notebook when tuning the predictor. Use the evaluator notebook afterward to measure the result.

In [ ]:
for window in result["local_windows"]:
    if not window.get(
        "valid_for_segmentation",
        False,
    ):
        continue

    print(
        f"{window['center_time']:6.2f}s | "
        f"{window['candidate_bpm']:6.1f} BPM | "
        f"grid offset="
        f"{1000 * window['reference_grid_offset']:+7.2f} ms | "
        f"confidence={window['confidence']:.3f}"
    )